# EduTrace — reproduction notebook

This notebook is the second of the two files Methods M19 names as the reproduction path. The first is `EduTrace_Revised_Pipeline.py`.

It runs top to bottom from a clean clone with no path editing. Every path below is relative to the repository root, so nothing reads a mounted Drive.

**Two modes.** By default the notebook *reads the locked artefacts* in `results/` and displays every reported quantity, which takes seconds. Setting `REGENERATE = True` re-runs the full pipeline from `data/` instead (about 25 minutes on CPU) and then displays the same quantities from the regenerated file. Both paths read the same committed inputs and land on the same numbers; the default exists so a reviewer can check the reconciliation without waiting for TabTransformer to train.

In [1]:
import json, os, subprocess, sys

# Repository root, whether the notebook is opened from notebooks/ or from the root.
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
os.chdir(ROOT)
sys.path.insert(0, ROOT)

REAL    = os.path.join('data', 'real_student_data_CLEANED_school.csv')
SYNTH   = os.path.join('data', 'synth_ctgan_s42.csv')
RESULTS = os.path.join('results', 'results.json')

REGENERATE = False   # True -> re-run the pipeline instead of reading the locked run

print('repository root :', ROOT)
for p in (REAL, SYNTH):
    print('%-42s exists=%s' % (p, os.path.exists(p)))

repository root : /home/claude/work/repo
data/real_student_data_CLEANED_school.csv  exists=True
data/synth_ctgan_s42.csv                   exists=True


## 1. The provenance guard (NF-1, Q9)

`assert_test_partition_is_real()` lives in the load path of `EduTrace_Revised_Pipeline.py`. It raises on any synthetic row in the evaluation partition. The self-test below also runs the negative control — it injects a synthetic row into the test period and confirms the guard fires — because an assertion that never fires on clean data proves nothing.

In [2]:
print(subprocess.run([sys.executable, 'selftest_nf1.py'],
                     capture_output=True, text=True).stdout)

NF-1 SELF-TEST — provenance guard

(a) clean load: the guard runs and the partition is what M9 states
NF-1 GUARD: test partition n=248 | synthetic rows=0 | real rows=248 | positives=7 (2.82%) — PASS
  [PASS] test partition prints 248 rows — got 248
  [PASS] test partition holds 0 synthetic rows — got 0
  [PASS] test partition holds at least one positive — got 7
  [PASS] the guard emitted its run-log line — NF-1 GUARD: test partition n=248 | synthetic rows=0 | real rows=248 | positives=7 (2.82%) — PASS

(b) Q9: the synthesiser is bounded to the training period
  [PASS] synthetic max academic_year < 2025 — max = 2024
  [PASS] synthetic rows in a test-period year == 0 — got 0

(c) negative control: inject one synthetic row into the test period
      guard raised as required:
      NF-1 GUARD FAILED: 1 synthetic row(s) in the test partition of 249. Every metric computed on this partition is contaminated. The synthesiser must be bounded to academic_year < 2025 (Q9).
  [PASS] guard RAISES on

## 2. The locked run

Reads `results/results.json`, or regenerates it when `REGENERATE` is set.

In [3]:
if REGENERATE:
    subprocess.run([sys.executable, 'EduTrace_Revised_Pipeline.py',
                    '--real', REAL, '--synth', SYNTH, '--out', RESULTS], check=True)

R = json.load(open(RESULTS))
print('keys in the results archive:', len(R))
print()
print('train+validation pool :', R['pool'])
print('evaluation partition  :', R['test'])

keys in the results archive: 29

train+validation pool : {'n': 530, 'pos': 62, 'neg': 468, 'ratio': 7.55, 'pct_pos': 11.7}
evaluation partition  : {'n': 248, 'pos': 7, 'neg': 241, 'pct_pos': 2.82}


## 3. Table 3 — held-out performance, mean ± SD across five seeds

The base rate is printed beside the metrics (clearance condition C5): AUC-PR is bounded below by class prevalence, so a level is not interpretable without it.

In [4]:
base = R['test']['pct_pos'] / 100.0
print('test-set base rate: %.4f  (%d positives of %d)\n'
      % (base, R['test']['pos'], R['test']['n']))
print('%-24s %-18s %-18s %-18s' % ('model', 'AUC-PR', 'AUC-ROC', 'recall'))
for k, v in R['table3'].items():
    print('%-24s %.4f +- %.4f   %.4f +- %.4f   %.4f +- %.4f'
          % (k, v['auc_pr']['mean'], v['auc_pr']['std'],
             v['auc_roc']['mean'], v['auc_roc']['std'],
             v['recall']['mean'], v['recall']['std']))
print()
print('attendance rule (binary)   single-point AUC-ROC %.4f | recall %.4f | precision %.4f'
      % (R['attendance_binary_rule']['single_point_auc_roc'],
         R['attendance_binary_rule']['recall'],
         R['attendance_binary_rule']['precision']))
print('attendance ranker (cont.)  AUC-ROC %.4f | AUC-PR %.4f'
      % (R['attendance_continuous_ranker']['auc_roc'],
         R['attendance_continuous_ranker']['auc_pr']))
print()
print('seed-42 thresholds:', R['seed42_thresholds'])
print('seed-42 scale_pos_weight:', R['scale_pos_weight_seed42'])

test-set base rate: 0.0282  (7 positives of 248)

model                    AUC-PR             AUC-ROC            recall            
xgb_default              0.1469 +- 0.0896   0.6816 +- 0.0422   0.3714 +- 0.1143
xgb_engineered           0.1167 +- 0.0444   0.6182 +- 0.0754   0.4000 +- 0.1069
decision_tree            0.0774 +- 0.0205   0.5914 +- 0.0448   0.3143 +- 0.0571
tabtransformer           0.2387 +- 0.0559   0.7248 +- 0.0409   0.5143 +- 0.0700

attendance rule (binary)   single-point AUC-ROC 0.7513 | recall 0.7143 | precision 0.0893
attendance ranker (cont.)  AUC-ROC 0.8678 | AUC-PR 0.1284

seed-42 thresholds: {'xgb_engineered': 0.34, 'xgb_default': 0.08, 'decision_tree': 0.6, 'tabtransformer': 0.5}
seed-42 scale_pos_weight: 7.49


## 4. NF-1 — the restored real-only re-scoring

An earlier draft of M10 retired this analysis as redundant. It is computed here rather than asserted. Under a correctly bounded synthesiser the two rows are identical, because the real-only subset *is* the evaluation partition — and that identity, computed, is the evidence.

In [5]:
nf1 = R['nf1_real_only_rescore']
for label, row in nf1['rows'].items():
    print('%-22s n=%3d  positives=%d  base=%.4f  AUC-PR=%.4f  recall=%.4f  precision=%.4f'
          % (label, row['n'], row['positives'], row['base_rate'],
             row['auc_pr'], row['recall'], row['precision']))
print()
print('subsets identical      :', nf1['subsets_identical'])
print('synthetic rows removed :', nf1['n_synthetic_removed'])
print()
for line in R['nf1_guard_log']:
    print(line)

full_test_partition    n=248  positives=7  base=0.0282  AUC-PR=0.1033  recall=0.2857  precision=0.0351
real_only_subset       n=248  positives=7  base=0.0282  AUC-PR=0.1033  recall=0.2857  precision=0.0351

subsets identical      : True
synthetic rows removed : 0

NF-1 GUARD: test partition n=248 | synthetic rows=0 | real rows=248 | positives=7 (2.82%) — PASS
NF-1 GUARD: test partition n=248 | synthetic rows=0 | real rows=248 | positives=7 (2.82%) — PASS
NF-1 GUARD: test partition n=151 | synthetic rows=0 | real rows=151 | positives=3 (1.99%) — PASS
NF-1 GUARD: test partition n=151 | synthetic rows=0 | real rows=151 | positives=3 (1.99%) — PASS


## 5. Q21 — the contribution-evidence anchor

RPS and DAS are reported under separate names (M17). RPS asks whether the packed alert names the model's own top-k SHAP features in order; DAS asks whether a top-k attribution carries a sign agreeing with the observed outcome. The original implementation computed the second and called it the first.

In [6]:
rps = json.load(open(os.path.join('results', 'rps_results.json')))
print('population        :', rps['population'])
print('threshold_used    :', rps['threshold_used'])
print('n_evaluated       :', rps['n_evaluated'])
print('true dropouts in that population:', rps['true_dropouts_in_population'])
print()
for block in ('rank_preservation_RPS', 'directional_agreement_DAS'):
    print(block)
    for k, v in rps[block].items():
        if k != 'definition':
            print('   %-26s %s' % (k, v))
print()
print('cross-model sensitivity:', {k: v for k, v in rps['cross_model_sensitivity'].items()
                                   if k != 'population'})
print()
print('Table 5 character compliance:', rps['table5_character_compliance'])
print()
for m in rps['example_alerts'][:2]:
    print('  (%d chars) %s' % (len(m), m))

population        : proposed model's own seed-42 flagged records
threshold_used    : 0.34
n_evaluated       : 57
true dropouts in that population: 2

rank_preservation_RPS
   rps_at_1                   1.0
   rps_at_2                   0.8596
   naive_fixed_order_at_1     0.193
   naive_fixed_order_at_2     0.0351
directional_agreement_DAS
   das_at_1                   0.7368
   das_at_2                   0.9825
   naive_fixed_order_at_1     0.0877
   naive_fixed_order_at_2     0.0877

cross-model sensitivity: {'n_evaluated': 40, 'rps_at_1': 1.0, 'rps_at_2': 0.875, 'naive_fixed_order_at_1': 0.175, 'naive_fixed_order_at_2': 0.025, 'das_at_1': 0.725, 'das_at_2': 0.925}

Table 5 character compliance: {'alerts': 57, 'within_160': 57, 'len_min': 70, 'len_max': 155, 'len_mean': 96.0, 'factors_min': 1, 'factors_max': 6, 'factors_mean': 2.46}

  (97 chars) ALERT:S01_2026 risk. Factors: lives far, male student, fees part-paid. Contact guardian.EduTrace.
  (112 chars) ALERT:S03_2025 risk. Factor

## 6. Q23 — the imbalance isolation grid, every cell

Sixteen cells: four SMOTE levels × two `scale_pos_weight` settings × two threshold arms. The spread is what is reported. No cell is selected.

In [7]:
import csv
with open(os.path.join('results', 'q23_imbalance_grid.csv')) as fh:
    grid = list(csv.DictReader(fh))
print('%9s %6s %13s %9s %8s %8s %8s' % ('smote', 'spw', 'thresh', 'AUC-PR',
                                        'recall', 'prec', 'F2'))
for r in grid:
    print('%9s %6s %13s %9s %8s %8s %8s'
          % (r['smote'], r['spw'], r['threshold_arm'], r['auc_pr'],
             r['recall'], r['precision'], r['f2']))
aps = [float(r['auc_pr']) for r in grid]
print('\nAUC-PR across the grid: min=%.4f max=%.4f range=%.4f'
      % (min(aps), max(aps), max(aps) - min(aps)))

    smote    spw        thresh    AUC-PR   recall     prec       F2
  removed    1.0   F2_selected    0.0871   0.2857      0.1   0.2083
  removed    1.0           0.5    0.0871   0.1429   0.1429   0.1429
  removed   7.49   F2_selected     0.076   0.1429   0.0909   0.1282
  removed   7.49           0.5     0.076   0.2857   0.0833   0.1923
      0.2    1.0   F2_selected    0.1233   0.2857      0.1   0.2083
      0.2    1.0           0.5    0.1233   0.2857      0.2   0.2632
      0.2   7.49   F2_selected    0.1033   0.2857   0.0351   0.1176
      0.2   7.49           0.5    0.1033   0.2857   0.0714   0.1786
      0.4    1.0   F2_selected     0.152   0.5714   0.0444   0.1695
      0.4    1.0           0.5     0.152   0.2857   0.1667     0.25
      0.4   7.49   F2_selected    0.1043   0.4286   0.0395   0.1442
      0.4   7.49           0.5    0.1043   0.2857   0.0357    0.119
      0.5    1.0   F2_selected    0.1534   0.5714   0.0381   0.1504
      0.5    1.0           0.5    0.1534   0.285

## 7. Q24 — rollback and temporal integrity

Each remediation layer reverted to its pre-remediation setting, one at a time, at the cut the Method declares (2025). Results that run against the study's own design choices are printed here rather than summarised away.

In [8]:
for name in ('q24_rollback_comparison.csv', 'q24_cut_comparison.csv'):
    print('---', name)
    with open(os.path.join('results', name)) as fh:
        for row in csv.DictReader(fh):
            print('   ', {k: row[k] for k in list(row)[:8]})
    print()

--- q24_rollback_comparison.csv
    {'arm': 'as-submitted (all fixes)', 'smote_k': '3', 'smote_strategy': '0.2', 'scale_pos_weight': '7.49', 'threshold': '0.34', 'auc_pr': '0.1033', 'macro_f1': '0.4618', 'precision': '0.0351'}
    {'arm': 'rollback SMOTE k 3 -> 5', 'smote_k': '5', 'smote_strategy': '0.2', 'scale_pos_weight': '7.49', 'threshold': '0.13', 'auc_pr': '0.1044', 'macro_f1': '0.3531', 'precision': '0.0376'}
    {'arm': 'rollback threshold F2 -> fixed 0.62', 'smote_k': '3', 'smote_strategy': '0.2', 'scale_pos_weight': '7.49', 'threshold': '0.62', 'auc_pr': '0.1033', 'macro_f1': '0.5496', 'precision': '0.1'}
    {'arm': 'rollback scale_pos_weight -> 1.0', 'smote_k': '3', 'smote_strategy': '0.2', 'scale_pos_weight': '1.0', 'threshold': '0.31', 'auc_pr': '0.1233', 'macro_f1': '0.5496', 'precision': '0.1'}
    {'arm': 'rollback SMOTE removed', 'smote_k': '3', 'smote_strategy': '', 'scale_pos_weight': '7.49', 'threshold': '0.74', 'auc_pr': '0.076', 'macro_f1': '0.5388', 'precision'

## 8. The clearance certificate and the null state

C1–C6 and the null state, from `closeout.py`. The null state governs what may be written: only state 4 licenses a positive finding.

In [9]:
V = json.load(open(os.path.join('results', 'closeout_verdict.json')))
print('%-6s %-44s %s' % ('ITEM', 'CHECK', 'VERDICT'))
for r in V['rows']:
    print('%-6s %-44s %s' % (r['item'], r['check'], r['verdict']))
print()
for c, state in V['clearance'].items():
    print('  %s  %s' % (c, state))
print()
print('NULL STATE : %d' % V['null_state'])
print('EVIDENCE   : %s' % V['null_evidence'])
print()
s = V['v1a_summary']
print('ten-seed delta vs the attendance ranker: %+.4f +- %.4f  '
      '(%d positive / %d negative, %d sign flips)'
      % (s['delta_mean'], s['delta_sd'], s['n_positive'], s['n_negative'],
         s['sign_flips']))
c3 = V['v1c']
print('paired MDE: %d discordant pairs observed, split %.3f; %d would be required'
      % (c3['n_discordant'], c3['observed_split'], c3['discordant_pairs_required']))

ITEM   CHECK                                        VERDICT
NF-1   assert in pipeline + passes                  CLOSED
Q9     synthesiser bounded to train period          CLOSED
Q21    RPS artefact traces to run                   CLOSED
Q23    full isolation grid computed                 CLOSED
Q24    lag + rollback + perturbation                CLOSED
Q4     reproduction path from clone                 CLOSED

  C1  FAIL
  C2  FAIL
  C3  FAIL — the wording is INCONCLUSIVE at this n
  C4  PASS — parity table written and budgets equal among the tunable arms
  C5  PASS
  C6  PENDING — decided by commit history, not by this script

NULL STATE : 5
EVIDENCE   : MDE at n=248 on 69 discordant pairs exceeds the observed separation

ten-seed delta vs the attendance ranker: -0.0047 +- 0.0437  (2 positive / 8 negative, 2 sign flips)
paired MDE: 69 discordant pairs observed, split 0.551; 786 would be required


## 9. Reconciliation

The quantities printed above are the ones Results R11 / Table 7 reconciles. Every figure in the manuscript traces to `results/results.json` from this single locked run.